# 08. Preprocessing Pipelines

**Track:** ML  
**Goal:** Preventing leakage with train/test splits, imputers, scalers, encoders, column transformers, and pipelines.

This notebook isolates one topic so you can run, change, and reason about each operation. Read the explanation before executing a cell, then change at least one input and predict the result.


## Start here: orientation and prerequisites

This lesson focuses on **Preprocessing Pipelines**. In plain language: Preventing leakage with train/test splits, imputers, scalers, encoders, column transformers, and pipelines. The topic belongs to the **ML** track, so interpret every operation through this larger foundation: Machine learning estimates a reusable mapping from examples. Generalization matters more than memorizing training data, so preprocessing, fitting, selection, and final evaluation must respect strict data boundaries.

### Prerequisites

Python, NumPy/pandas, train-test splitting, descriptive statistics, and the distinction between features and targets.

### Complete learning path

`problem framing -> split -> preprocessing -> baseline -> fit -> validate -> tune -> test -> error analysis`

### Evidence of understanding

Held-out metrics against a baseline, cross-validation variability, segment-level errors, calibration or ranking diagnostics, and reproducible pipelines.

If a prerequisite is unfamiliar, read the earlier numbered notebooks first, but you can still use this notebook as a glossary-driven standalone reference.


## Learning objectives

By the end, you should be able to:

1. Define the technique and explain why it exists.
2. Describe its inputs, outputs, assumptions, and computational tradeoffs.
3. Use the main APIs without copying blindly.
4. Interpret intermediate and final outputs.
5. Diagnose common failure modes and leakage.
6. Compare the technique with a reasonable baseline.
7. Apply the feature to a small new problem.
8. Explain how it fits into a production system.

> Run cells from top to bottom. All examples are deterministic where randomness is used.


## Concept and intuition

Preventing leakage with train/test splits, imputers, scalers, encoders, column transformers, and pipelines.

Think of the workflow as a contract: an input with known shape and meaning enters, an operation applies explicit rules or learned parameters, and an output with a measurable interpretation leaves. The important habit is to separate **fit/learning operations** from **transform/prediction operations**. Any statistic or parameter learned from data must be learned from training data only. For analysis-only topics, preserve raw data and make transformations on a copy.

### Why this matters

A method is useful only when its assumptions match the problem. Before writing code, state the decision the output will support, the cost of errors, the baseline, and the evidence required to trust the result.


## Mathematical and computational view

Most data and AI operations can be described as a mapping $f: X \rightarrow Y$. In learned systems, parameters $\theta$ are estimated by minimizing an objective such as

$$\theta^* = \arg\min_\theta \; \mathcal{L}(f_\theta(X), y) + \lambda R(\theta).$$

Here $\mathcal{L}$ measures error, $R$ controls complexity, and $\lambda$ sets the regularization strength. Not every topic uses this exact objective, but the questions remain useful: what is learned, what is fixed, what is optimized, and what evidence evaluates success?

### Complexity questions

- How does runtime grow with rows, columns, classes, tokens, or stored vectors?
- Does the operation materialize a dense copy?
- Can fitting and inference be batched?
- Which state must be persisted for reproducibility?


## Under the hood: internal mechanics

A supervised learner receives feature-target pairs and searches a hypothesis space for parameters that reduce training loss. Inductive bias determines which relationships are easy to learn. Regularization, tree limits, neighborhood size, margins, or dimensionality constraints control effective complexity. Validation estimates how choices generalize; the untouched test set estimates the final selected workflow, including preprocessing.

### Trace the state transition

For **Preprocessing Pipelines**, write the state before the central operation, the state learned or changed by it, and the state afterward. Distinguish fixed configuration from data-derived state. If no fitting occurs, identify which information the deterministic transformation preserves, aggregates, reorders, or discards.

A useful tracing table is:

| Stage | Input | Operation | State created or used | Output |
|---|---|---|---|---|
| Prepare | Raw values | Validate/encode | Schema and configuration | Model-ready representation |
| Core | Prepared input | Preventing leakage with train/test splits, imputers, scalers, encoders, column transformers, and pipelines. | Fixed or learned state | Intermediate result |
| Decide | Intermediate result | Threshold/rank/format | Decision policy | Final output |
| Evaluate | Output + reference | Metric/error analysis | Evaluation configuration | Evidence |

Complete this table using the actual variables in the code cells.


## Alternatives and tradeoffs

Begin with a constant or rule baseline. Use linear models for transparent additive effects, trees for nonlinear thresholds and interactions, ensembles for strong tabular performance, nearest neighbors for local similarity, support-vector methods for margin-based boundaries, clustering when labels are absent, and PCA when compression of correlated numeric features is justified.

### Comparison dimensions

Never compare techniques only by one quality score. Compare:

- Data and label requirements
- Assumptions and inductive bias
- Interpretability and auditability
- Training/build cost and inference/query cost
- Memory, latency, and throughput
- Robustness to missing, rare, shifted, or adversarial inputs
- Update frequency and maintenance burden
- Privacy, permissions, and safety impact

The preferred technique is the least complex option that satisfies the real quality and operational constraints.


## Inputs and pre-flight checks

Before applying this feature:

- Confirm row meaning, feature meaning, units, shape, and dtype.
- Quantify missing, duplicate, invalid, extreme, and out-of-domain values.
- Identify target leakage and time leakage.
- Decide the train/validation/test boundary before fitting anything.
- Set random seeds and record library versions.
- Define a baseline and primary metric.
- Preserve identifiers and source metadata needed for debugging.


## Worked decision scenario

For churn prediction, split customers rather than rows if one customer appears repeatedly. Fit imputers and encoders inside the training pipeline, compare logistic regression with a tree ensemble, tune on cross-validation, choose a threshold from retention cost, inspect false negatives by customer segment, and preserve the final test set for one unbiased estimate.

### Decision record to complete

1. **User and decision:** Who consumes the output, and what action follows?
2. **Unit of analysis:** What exactly does one row, item, query, or request represent?
3. **Baseline:** What simple behavior must be beaten?
4. **Primary failure cost:** Which wrong output causes the greatest harm or expense?
5. **Evaluation:** Which metric, slice, and example set reveal that failure?
6. **Operational constraints:** What latency, cost, freshness, privacy, and availability limits apply?
7. **Fallback:** What happens when input is invalid, confidence is low, evidence is missing, or the system fails?

This decision record should be written before tuning the implementation.


## Example 1: Build and inspect

The next cell creates a compact example. Inspect shapes, types, and values before applying more operations.


In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X = pd.DataFrame({"age": [22, 35, None, 44, 29], "income": [35, 70, 52, 90, 48], "city": ["A", "B", "A", "C", None]})
numeric = Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())])
categorical = Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("encode", OneHotEncoder(handle_unknown="ignore"))])
preprocessor = ColumnTransformer([("num", numeric, ["age", "income"]), ("cat", categorical, ["city"])])
transformed = preprocessor.fit_transform(X)
print(transformed.toarray() if hasattr(transformed, "toarray") else transformed)


## Worked-example walkthrough

Read the preceding code in four passes instead of treating it as one block:

1. **Imports:** identify which behavior comes from Python and which comes from an external library.
2. **Data construction:** locate each input, its type, dimensions, units, and semantic meaning.
3. **Operation:** find the line that performs the central transformation, fit, retrieval, or calculation.
4. **Inspection:** explain why each printed value is useful and what an unexpected value would mean.

### API sequence used by this topic

1. **`train_test_split`**: Creates independent training and test partitions.
2. **`SimpleImputer`**: Learns replacement values from training data.
3. **`StandardScaler`**: Centers and scales numeric features.
4. **`OneHotEncoder`**: Encodes nominal categories safely.
5. **`Pipeline/ColumnTransformer`**: Chains transformations by column and prevents leakage.

For every API, say the input and return value aloud. Then use `type(...)`, `getattr(value, "shape", None)`, and `help(...)` to verify your statement.


## Example 2: Apply the feature

This cell demonstrates the main workflow. Change one parameter at a time and compare the output; that makes cause and effect visible.


In [ ]:
print(preprocessor.get_feature_names_out())
print("Important rule: fit preprocessing only on training data; use transform on validation/test data.")


## Solved reasoning exercise

**Question:** How should a learner decide whether the preceding result is trustworthy?

**Answer:** First verify that the input contract is satisfied and no future, target, or unauthorized information entered the operation. Next compare the result with a simple baseline and inspect more than one metric or example. Check sensitivity to a meaningful parameter, seed, threshold, or input perturbation. Finally, test a boundary case and explain the result in domain language. Trust is accumulated from these checks; it is not created by a cell executing without an exception.

**Question:** What should be saved for reproduction?

**Answer:** Save the input or its immutable version, split identifiers, preprocessing state, parameters, random seed, code and dependency versions, learned artifact or index version, metric definitions, and representative outputs. For LLM/RAG/agent work, also save prompts, decoding settings, corpus version, retrieval configuration, tool schemas, and evaluator versions.


## Failure-mode analysis

Analyze failures at four levels:

### 1. Data or context failure

The input may be missing, stale, duplicated, mislabeled, biased, unauthorized, malformed, or outside the range represented during development. Validate before the core operation and preserve enough identifiers to trace the source.

### 2. Method failure

The technique may have the wrong inductive bias, insufficient capacity, excessive complexity, unstable optimization, poor parameters, weak retrieval, or a mismatched objective. Compare with controlled alternatives and inspect intermediate state.

### 3. Evaluation failure

A convenient metric may hide costly errors, leakage, uncertainty, subgroup degradation, unsupported claims, or changing prevalence. Rebuild the evaluation around the real decision and representative cases.

### 4. System failure

Schemas, dependencies, permissions, services, indexes, timeouts, or monitoring may fail even when the algorithm is correct. Test the complete path and define fallback behavior.

For each observed error, assign one primary category, record evidence, and choose a fix that targets its root cause.


## Function and API reference

### `train_test_split`

Creates independent training and test partitions.

**Inspect carefully:** accepted inputs, shape requirements, defaults, return type, learned attributes, error behavior, and computational cost.

### `SimpleImputer`

Learns replacement values from training data.

**Inspect carefully:** accepted inputs, shape requirements, defaults, return type, learned attributes, error behavior, and computational cost.

### `StandardScaler`

Centers and scales numeric features.

**Inspect carefully:** accepted inputs, shape requirements, defaults, return type, learned attributes, error behavior, and computational cost.

### `OneHotEncoder`

Encodes nominal categories safely.

**Inspect carefully:** accepted inputs, shape requirements, defaults, return type, learned attributes, error behavior, and computational cost.

### `Pipeline/ColumnTransformer`

Chains transformations by column and prevents leakage.

**Inspect carefully:** accepted inputs, shape requirements, defaults, return type, learned attributes, error behavior, and computational cost.

Use `help(function_name)`, `inspect.signature(function_name)`, or append `?` in Jupyter to inspect the installed-version signature and documentation. Never assume an online example matches your installed version.


## Parameter study

| Function or concept | Primary responsibility | Experiment |
|---|---|---|
| `train_test_split` | Creates independent training and test partitions. | Start with defaults, then vary one argument. |
| `SimpleImputer` | Learns replacement values from training data. | Compare output before and after changing it. |
| `StandardScaler` | Centers and scales numeric features. | Compare output before and after changing it. |
| `OneHotEncoder` | Encodes nominal categories safely. | Compare output before and after changing it. |
| `Pipeline/ColumnTransformer` | Chains transformations by column and prevents leakage. | Compare output before and after changing it. |

For each experiment, predict the direction of change before running it. Record quality, runtime, memory, and stability rather than choosing a setting from one score.


In [ ]:
# Uncomment one line at a time to explore documentation.
# help(type)
# help(print)
print("Use help(...) on any function or class introduced above.")


## Diagnostics and interpretation

A final score is not a diagnosis. Inspect intermediate representations, fitted attributes, residuals or errors, and performance by meaningful segment. Compare training and validation behavior to identify underfitting or overfitting. Repeat evaluation across seeds or folds when sampling variation matters.

### Questions to ask

- Is the output calibrated and stable?
- Which observations produce the largest errors?
- Does performance degrade for rare categories or long-tail queries?
- Is the result sensitive to scale, initialization, ordering, or threshold?
- Can a simpler baseline match it?
- Does the output support the actual decision rather than a proxy?


## Debugging laboratory

Use this sequence whenever the example fails or produces a surprising answer:

1. Restart the kernel and run cells in order to eliminate hidden state.
2. Read the complete traceback from the final line upward.
3. Print input types, shapes, column names, ranges, null counts, and a few values.
4. Reduce the failure to the smallest input that still reproduces it.
5. Check installed API signatures with `help(...)`; do not guess from another version.
6. Test one hypothesis at a time and add an assertion when the cause is found.
7. Classify the root cause as data, API contract, algorithm, evaluation, environment, permission, or system behavior.

### Typical diagnosis table

| Symptom | Likely question | First check |
|---|---|---|
| Shape or length error | Are rows/features aligned? | Print every relevant `.shape` |
| Missing or unknown label | Did train and inference vocabularies differ? | Compare columns/categories |
| Unrealistically high score | Is there leakage or duplicate overlap? | Rebuild split before preprocessing |
| Unstable result | Is randomness or sample size dominating? | Fix seeds and repeat runs |
| Slow execution | Which step scales poorly? | Time stages separately |
| Unsupported answer/action | Was evidence or authorization absent? | Inspect retrieved context and policy gate |


## Common mistakes and remedies

- **Unchecked inputs:** validate shape, dtype, units, missingness, and domain constraints.
- **Leakage:** split first; fit preprocessing only inside the training workflow.
- **Training-only evaluation:** reserve unseen data and use cross-validation where appropriate.
- **Wrong metric:** connect the metric to false-positive, false-negative, latency, and business costs.
- **Causal overclaim:** association, coefficients, attention, and importance do not prove causality.
- **Uncontrolled experiments:** change one factor at a time and record seeds and versions.
- **Ignoring uncertainty:** report variability, confidence intervals, or sensitivity analysis.
- **Missing baseline:** compare against a naive, rule-based, or simpler statistical method.
- **Deployment mismatch:** package transformations with the model and test unseen inputs.
- **No monitoring:** define drift, quality, latency, cost, and failure alerts before release.


## Production and engineering notes

A notebook proves an idea; a production component needs explicit schemas, validation, tests, versioned artifacts, deterministic preprocessing, structured logs, latency budgets, access controls, and rollback behavior. Persist every learned preprocessing object together with the model. For RAG, also version the source corpus, chunker, embedding model, index parameters, prompt, and evaluation set.

### Minimum artifact record

- Data or corpus version and split policy
- Code commit and dependency versions
- Parameters, random seed, and fitted artifact identifier
- Evaluation metrics with segment breakdowns
- Known limitations and unsafe input conditions
- Owner, monitoring thresholds, and rollback procedure


## Guided practice: beginner to advanced

### Level 1: Observe
1. Run the example and explain every printed value.
2. Write down the shape and dtype after each operation.
3. Use `help(...)` on every introduced function.

### Level 2: Modify
4. Replace the sample data with at least twice as many rows.
5. Change one important parameter and explain the difference.
6. Add a missing value, unseen category, extreme number, empty text, or imbalance.

### Level 3: Engineer
7. Write a typed reusable function with a docstring and input validation.
8. Add numerical assertions and at least three edge-case tests.
9. Compare against a baseline using an appropriate metric.

### Level 4: Critique
10. Identify one violated assumption and demonstrate its effect.
11. Measure sensitivity across parameters or random seeds.
12. Explain when this technique should not be used and propose an alternative.


In [ ]:
# Your practice area
# Write your solution here, then add assertions below.



## Mini-project challenge

Build a small end-to-end application of this topic. Include a written problem statement, input contract, exploratory checks, baseline, implementation, evaluation, error analysis, and conclusion. Your conclusion must separate observed evidence from assumptions. Add a `README` section describing how another learner can reproduce the result.

### Definition of done

- The notebook runs top to bottom from a clean kernel.
- All randomness is controlled.
- Every non-obvious function and parameter is explained.
- Tests include normal, boundary, and invalid inputs.
- Results include a baseline and at least one diagnostic beyond the headline metric.
- Limitations, ethical risks, and next steps are explicit.


## Where this topic connects next

Package preprocessing and prediction together, persist artifacts, monitor drift, and retrain only through a controlled evaluation gate.

### Transfer questions

- Which ideas remain valid if the library or model changes?
- Which parts are domain assumptions rather than technical facts?
- How would the workflow change for streaming data, very large inputs, strict latency, or sensitive information?
- Which output must a human review before action?
- What simpler deterministic rule could serve as a fallback?

A learner has mastered the topic when they can transfer the reasoning to new data and APIs, not merely reproduce the exact sample output.


## Knowledge check and interview questions

1. What information is learned from data and where is that state stored?
2. Which inputs and outputs does each key function expect?
3. What assumption is easiest to violate?
4. Which metric or visual would expose a poor result?
5. How would this step fit into a reproducible pipeline?
6. What changes when the dataset becomes 100 times larger?
7. How would you debug a train/production discrepancy?
8. Which simpler alternative should be tested first?
9. How would you explain the result to a non-technical stakeholder?
10. What monitoring signal would reveal degradation first?


## Glossary

- **Feature:** an input variable available to an analysis or model.
- **Target:** the outcome a supervised model learns to predict.
- **Parameter:** a value learned from data.
- **Hyperparameter:** a configuration chosen outside model fitting.
- **Fit:** learn state from training observations.
- **Transform:** apply a learned or fixed mapping.
- **Inference:** produce outputs for new inputs.
- **Baseline:** a simple reference method.
- **Generalization:** performance on unseen data.
- **Leakage:** information unavailable at real prediction time entering training.
- **Drift:** production input or relationship changes over time.
- **Reproducibility:** ability to obtain the same result from recorded inputs and settings.


## Takeaways

- Preventing leakage with train/test splits, imputers, scalers, encoders, column transformers, and pipelines.
- Inspect data before and after every transformation.
- Keep experiments reproducible and validation separate.
- The matching guide in `../Docs/08-preprocessing-pipelines.md` contains deeper theory, API notes, and further exercises.
